# SPARK Pipeline — Google Colab A100/H100
Exécuter les cellules dans l'ordre. Prévoir ~15-20 min pour un run complet.

## 0. Vérification GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
# Doit afficher A100 ou H100 — sinon: Runtime > Change runtime type > A100

## 1. Cloner le repo

In [ ]:
import os
REPO_URL = 'https://github.com/Blockprod/SPARK.git'
PROJECT_DIR = '/content/SPARK'
LTX2_DIR = '/content/LTX-2'

# Clone SPARK
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} checkout -- .
    !git -C {PROJECT_DIR} pull

# Clone LTX-2 (official repo — needed for ltx-core and ltx-pipelines packages)
if not os.path.exists(LTX2_DIR):
    !git clone https://github.com/Lightricks/LTX-2.git {LTX2_DIR}
else:
    !git -C {LTX2_DIR} pull

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())


## 2. Installer les dépendances

In [ ]:
# Install ltx-core + ltx-pipelines from the cloned LTX-2 repo (NOT diffusers)
!pip install -q -e /content/LTX-2/packages/ltx-core -e /content/LTX-2/packages/ltx-pipelines

# Install remaining SPARK dependencies (no diffusers/transformers needed)
!pip install -q \
    pydantic-settings PyYAML python-dotenv httpx tenacity \
    google-genai \
    safetensors huggingface-hub \
    edge-tts \
    kokoro-onnx==0.4.9 onnxruntime-gpu soundfile librosa \
    ffmpeg-python pysubs2 opencv-python Pillow numpy \
    pytrends praw \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib \
    APScheduler uvicorn sse-starlette jinja2 python-multipart \
    rich cryptography

!apt-get install -qq ffmpeg
print('Installation terminée (LTX-2.3 native pipeline).')


## 3. Monter Google Drive et copier les secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SECRETS = '/content/drive/MyDrive/SPARK_secrets'
import shutil, os

shutil.copy(f'{DRIVE_SECRETS}/.env', '/content/SPARK/.env')
os.makedirs('/content/SPARK/secrets', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/client_secret.json', '/content/SPARK/secrets/client_secret.json')
shutil.copy(f'{DRIVE_SECRETS}/youtube_token.json', '/content/SPARK/secrets/youtube_token.json')
os.makedirs('/content/SPARK/models', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/models/kokoro-v1.0.fp16.onnx', '/content/SPARK/models/kokoro-v1.0.fp16.onnx')
shutil.copy(f'{DRIVE_SECRETS}/models/voices-v1.0.bin', '/content/SPARK/models/voices-v1.0.bin')
print('Secrets copiés ✓')

## 4. Connexion HuggingFace (pour LTX-Video)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace connecté ✓')

In [ ]:
import psutil, shutil
ram_gb = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage('/')
disk_free_gb = disk.free / (1024**3)
print(f'RAM système : {ram_gb:.1f} Go')
print(f'Disque libre : {disk_free_gb:.1f} Go')
if ram_gb < 40:
    print('⚠️  RAM < 40 Go — risque OOM au chargement du modèle LTX-2.3 (22B) !')
    print('   → Runtime > Change runtime type > cocher "High-RAM"')
else:
    print('✓ RAM suffisante pour LTX-2.3')
if disk_free_gb < 60:
    print('⚠️  Disque < 60 Go — les modèles LTX-2.3 (~50 Go) risquent de ne pas tenir.')
    print('   → Utiliser le cache Drive pour éviter le re-téléchargement')
else:
    print('✓ Espace disque OK')


## 5a. Télécharger les modèles LTX-2.3

**Prérequis (1 seule fois)** : créer le bucket GCS depuis [Cloud Console](https://console.cloud.google.com/storage/browser?project=spark-492115) → "Create bucket" → nom `spark-ltx2-models-492115`, région `us-central1`, stockage standard.


In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download
from google.colab import auth

# ── Configuration ─────────────────────────────────────────────────────────
# Après avoir créé le bucket GCS une fois, mettre USE_GCS_CACHE = True.
# Stockage ~50 Go ≈ 1$/mois sur GCS (project spark-492115).
# Premier run : télécharge depuis HF + upload GCS (~10 min total).
# Runs suivants : copie GCS → Colab (~1-2 min via réseau Google interne).
USE_GCS_CACHE = True
GCS_BUCKET = 'gs://spark-ltx2-models-492115/ltx2'
# ──────────────────────────────────────────────────────────────────────────

LOCAL_MODELS = '/content/models/ltx2'
os.makedirs(LOCAL_MODELS, exist_ok=True)

# Auth GCS (utilise le compte Google connecté à Colab)
if USE_GCS_CACHE:
    auth.authenticate_user()

def gcs_exists(gcs_path):
    """Vérifie si un objet/préfixe existe dans GCS."""
    result = os.popen(f'gsutil -q stat {gcs_path} 2>&1; echo $?').read().strip()
    # gsutil ls renvoie 0 si trouvé
    rc = os.popen(f'gsutil ls {gcs_path} > /dev/null 2>&1; echo $?').read().strip()
    return rc == '0'

def ensure_model(repo_id, filename, local_dir):
    local_path = os.path.join(local_dir, filename)
    gcs_path = f'{GCS_BUCKET}/{filename}'

    # 1. Déjà présent localement (vraie taille > 1 Mo)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1024 * 1024:
        size_gb = os.path.getsize(local_path) / (1024**3)
        print(f'  ✓ Local ({size_gb:.1f} Go): {filename}')
        return local_path

    # 2. Copie depuis GCS (rapide — même réseau Google que Colab)
    if USE_GCS_CACHE:
        rc = os.system(f'gsutil -q stat {gcs_path}')
        if rc == 0:
            print(f'  ↗ Copie GCS → local: {filename}')
            os.system(f'gsutil -m cp {gcs_path} {local_path}')
            return local_path

    # 3. Téléchargement depuis HuggingFace (fallback, ~3-5 min)
    print(f'  ⬇ Téléchargement HF: {repo_id}/{filename}')
    downloaded = hf_hub_download(
        repo_id=repo_id, filename=filename,
        local_dir=local_dir, local_dir_use_symlinks=False
    )

    # Upload vers GCS pour les sessions suivantes
    if USE_GCS_CACHE:
        print(f'  ☁️  Upload GCS: {filename}')
        os.system(f'gsutil -m cp {downloaded} {gcs_path}')

    return downloaded

# 1. DiT checkpoint (~44 Go)
print('Modèle DiT 22B...')
ensure_model('Lightricks/LTX-2.3', 'ltx-2.3-22b-distilled-1.1.safetensors', LOCAL_MODELS)

# 2. Spatial upscaler (~2 Go)
print('Spatial upscaler 2x...')
ensure_model('Lightricks/LTX-2.3', 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors', LOCAL_MODELS)

# 3. Gemma 3 12B text encoder (~5 Go)
GEMMA_LOCAL = os.path.join(LOCAL_MODELS, 'gemma-3-12b-it-qat-q4_0-unquantized')
GCS_GEMMA = f'{GCS_BUCKET}/gemma-3-12b-it-qat-q4_0-unquantized'

if os.path.exists(GEMMA_LOCAL) and any(f.endswith('.safetensors') for f in os.listdir(GEMMA_LOCAL)):
    print(f'  ✓ Gemma 3 local: {GEMMA_LOCAL}')
elif USE_GCS_CACHE and os.system(f'gsutil -q ls {GCS_GEMMA}/config.json') == 0:
    print('  ↗ Copie Gemma 3 depuis GCS...')
    os.makedirs(GEMMA_LOCAL, exist_ok=True)
    os.system(f'gsutil -m cp -r {GCS_GEMMA}/* {GEMMA_LOCAL}/')
else:
    print('  ⬇ Téléchargement Gemma 3 12B depuis HF...')
    snapshot_download(
        'google/gemma-3-12b-it-qat-q4_0-unquantized',
        local_dir=GEMMA_LOCAL, local_dir_use_symlinks=False
    )
    if USE_GCS_CACHE:
        print('  ☁️  Upload Gemma 3 vers GCS...')
        os.system(f'gsutil -m rsync -r {GEMMA_LOCAL} {GCS_GEMMA}')

# Vérification finale
print()
for name, path in [
    ('DiT checkpoint', os.path.join(LOCAL_MODELS, 'ltx-2.3-22b-distilled-1.1.safetensors')),
    ('Spatial upscaler', os.path.join(LOCAL_MODELS, 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors')),
    ('Gemma 3 root', GEMMA_LOCAL),
]:
    if os.path.exists(path):
        print(f'✓ {name}')
    else:
        raise FileNotFoundError(f'{name} introuvable: {path}')

print('\nTous les modèles LTX-2.3 sont prêts ✓')


## 5b. Vérifier la config LTX-2.3


In [ ]:
import yaml
config_path = '/content/SPARK/config.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Force LTX-2 config values (in case git pull brought old version)
cfg['video_generation']['provider'] = 'ltx-2'
cfg['video_generation']['checkpoint_path'] = '/content/models/ltx2/ltx-2.3-22b-distilled-1.1.safetensors'
cfg['video_generation']['spatial_upsampler_path'] = '/content/models/ltx2/ltx-2.3-spatial-upscaler-x2-1.1.safetensors'
cfg['video_generation']['gemma_root'] = '/content/models/ltx2/gemma-3-12b-it-qat-q4_0-unquantized'
cfg['video_generation']['device'] = 'cuda'
cfg['video_generation']['fp8'] = True
cfg['video_generation']['scenes']['max_scene_duration_sec'] = 10
cfg['pipeline']['target_width'] = 576
cfg['pipeline']['target_height'] = 1024
cfg['pipeline']['target_fps'] = 25
# Subtitles — calibrated for 576x1024 portrait (do not change)
cfg['post_production']['subtitles']['style']['font_size'] = 48
cfg['post_production']['subtitles']['style']['bold'] = True
cfg['post_production']['subtitles']['style']['margin_v'] = 80
# Edge TTS configuration
if 'audio_generation' not in cfg:
    cfg['audio_generation'] = {}
cfg['audio_generation']['active_backend'] = 'edge_tts'
if 'edge_tts' not in cfg['audio_generation']:
    cfg['audio_generation']['edge_tts'] = {}
cfg['audio_generation']['edge_tts']['voice'] = 'fr-FR-DeniseNeural'
cfg['audio_generation']['edge_tts']['rate'] = '-5%'
cfg['audio_generation']['edge_tts']['pitch'] = '+0Hz'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

# Verification
with open(config_path, 'r') as f:
    check = yaml.safe_load(f)

v = check['video_generation']
p = check['pipeline']
print(f"  Provider          : {v['provider']}")
print(f"  Checkpoint        : {v['checkpoint_path']}")
print(f"  Spatial upsampler : {v['spatial_upsampler_path']}")
print(f"  Gemma root        : {v['gemma_root']}")
print(f"  FP8               : {v['fp8']}")
print(f"  Resolution        : {p['target_width']}x{p['target_height']}")
print(f"  FPS               : {p['target_fps']}")
print(f"  Sub style         : font={check['post_production']['subtitles']['style']['font_size']}pt "
      f"bold={check['post_production']['subtitles']['style']['bold']} "
      f"margin_v={check['post_production']['subtitles']['style']['margin_v']}")

if v['provider'] != 'ltx-2':
    raise RuntimeError(f"ERREUR: provider={v['provider']} au lieu de ltx-2 !")
if not v['fp8']:
    raise RuntimeError('ERREUR: fp8 doit être True pour A100-40GB !')

print('\nConfig LTX-2.3 vérifiée ✓ — prêt à lancer le pipeline')


## 6. Lancer le pipeline


In [ ]:
import os
TOPIC = "L'intelligence artificielle change le monde"
os.chdir('/content/SPARK')
!python pipeline.py --topic "{TOPIC}"


## 7. Télécharger le MP4 final


In [ ]:
import glob
from google.colab import files
renders = sorted(glob.glob('/content/SPARK/outputs/renders/*.mp4'))
if renders:
    final = renders[-1]
    print(f'Téléchargement: {final}')
    files.download(final)
else:
    print('Aucun fichier MP4 trouvé dans outputs/renders/')

# Optionnel: sauvegarder sur Drive
import shutil, os
DRIVE_OUTPUTS = '/content/drive/MyDrive/SPARK_outputs'
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
for mp4 in glob.glob('/content/SPARK/outputs/renders/*.mp4'):
    dest = os.path.join(DRIVE_OUTPUTS, os.path.basename(mp4))
    shutil.copy2(mp4, dest)
    print(f'Sauvegardé → {dest}')
